# TTAP: cold and greedy-initialized HiGHS

Companion to commit `07dd90042822e2cab84209fe81c9f31ecfe230a8`.
The objective remains the single-primary-output formulation. Original notebooks and CSVs are not overwritten.

Install `requirements.txt` in a separate environment before running this notebook. Start with the pilot; do not treat pilot output as the 30-replicate paper experiment.

In [ ]:
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), Path.cwd() / 'Resuelto_Paper' / 'TTAP_WarmStart', Path.cwd() / 'TTAP_WarmStart']
SCRIPT_DIR = next((p for p in candidates if (p / 'ttap_warm_start.py').is_file()), None)
if SCRIPT_DIR is None:
    raise FileNotFoundError('Open this notebook from the TTAP_WarmStart folder.')
print('Companion directory:', SCRIPT_DIR)


## Paired experiment

Begin with 20 tasks and one seed. Then use both configurations and 30 seeds per task count.
Every budget restarts the solver independently. Each treatment has the same native HiGHS version and options.
Set `RUN_EXPERIMENT=True` deliberately; a nonempty output directory is not silently overwritten.

In [ ]:
EXPERIMENTS = ['A']
TASKS = [20]
SEEDS = [1]
BUDGETS = [1, 5, 15, 60]
RUN_EXPERIMENT = False
OUTPUT = SCRIPT_DIR / 'runs' / 'notebook_pilot'

command = [sys.executable, str(SCRIPT_DIR / 'ttap_warm_start.py'),
           '--experiments', *EXPERIMENTS,
           '--tasks', *map(str, TASKS), '--seeds', *map(str, SEEDS),
           '--budgets', *map(str, BUDGETS), '--output', str(OUTPUT)]
print(' '.join(command))
if RUN_EXPERIMENT:
    subprocess.run(command, check=True)


## Figure 5 and optional warm-start plots

The current CSVs already permit plan-availability, incumbent-value, and certified-reference shortfall reporting.
New warm plots are generated only when a new metrics file exists.
Only A's 600-s and B's certified 1800-s optima are used as certified references.

In [ ]:
repo_paper = SCRIPT_DIR.parent
A_CSV = repo_paper / 'ttap_experiment_A_4x4_impact_anytime_traced' / 'metrics_A4x4_paper_20260808_112401-0400_matched_v3_impact_anytime.csv'
B_CSV = repo_paper / 'ttap_experiment_B_6x7_stress_impact_anytime_traced' / 'metrics_B6x7_stress_paper_20260825_195505-0400_matched_v3_impact_anytime_old6x7_operational_config.csv'
FIGURES = SCRIPT_DIR / 'figures' / 'notebook_analysis'
command = [sys.executable, str(SCRIPT_DIR / 'figure5_and_analysis.py'),
           '--a', str(A_CSV), '--b', str(B_CSV), '--output', str(FIGURES)]
if (OUTPUT / 'metrics.csv').is_file():
    command += ['--warm-csv', str(OUTPUT / 'metrics.csv')]
subprocess.run(command, check=True)


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(FIGURES / 'Figure5_current.png')))
for experiment in EXPERIMENTS:
    image = FIGURES / f'Figure_WarmStart_{experiment}.png'
    if image.is_file():
        display(Image(filename=str(image)))
